# 01 - Bronze to Silver
Nettoyage des donnees brutes et sauvegarde en format Parquet

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder \
    .appName("01_Bronze_to_Silver") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()

print("Spark connecte")

Spark connecte


In [2]:
# # Lire les donnees brutes depuis HDFS (Bronze)
trips_raw = spark.read.csv("hdfs://namenode:9000/projet/bronze/fact_trips_sample.csv",
                             header=True, inferSchema=True)
dates = spark.read.csv("hdfs://namenode:9000/projet/bronze/dim_data.csv",
                        header=True, inferSchema=True)
locations = spark.read.csv("hdfs://namenode:9000/projet/bronze/dim_localizacao.csv",
                             header=True, inferSchema=True)
weather = spark.read.csv("hdfs://namenode:9000/projet/bronze/dim_clima.csv",
                           header=True, inferSchema=True)

print(f"Trips: {trips_raw.count()} lignes")
print(f"Dates: {dates.count()} lignes")
print(f"Locations: {locations.count()} lignes")
print(f"Weather: {weather.count()} lignes")

trips_raw.show(5)                                                                                                                                                                
print("Lecture OK")  

Trips: 1000000 lignes
Dates: 366 lignes
Locations: 263 lignes
Weather: 38 lignes
+-------+----------------------+----------------------+--------+----------------+-----------+
|data_id|localizacao_partida_id|localizacao_chegada_id|clima_id|distancia_viagem|valor_total|
+-------+----------------------+----------------------+--------+----------------+-----------+
|  327.0|                   168|                   159|    NULL|            1.14|       16.8|
|  257.0|                    79|                    41|    NULL|            8.46|      52.44|
|  268.0|                    70|                    79|    NULL|             9.7|       69.7|
|  171.0|                   235|                   139|    NULL|            0.61|       12.5|
|   67.0|                   160|                   232|    NULL|            1.55|       19.8|
+-------+----------------------+----------------------+--------+----------------+-----------+
only showing top 5 rows

Lecture OK


In [3]:
# Traduire les noms de colonnes en anglais

# fact_trips
trips_raw = trips_raw \
    .withColumnRenamed("data_id", "date_id") \
    .withColumnRenamed("localizacao_partida_id", "pickup_location_id") \
    .withColumnRenamed("localizacao_chegada_id", "dropoff_location_id") \
    .withColumnRenamed("clima_id", "weather_id") \
    .withColumnRenamed("distancia_viagem", "trip_distance") \
    .withColumnRenamed("valor_total", "total_amount")

# dim_data
dates = dates \
    .withColumnRenamed("data_id", "date_id") \
    .withColumnRenamed("data", "date") \
    .withColumnRenamed("ano", "year") \
    .withColumnRenamed("mes", "month") \
    .withColumnRenamed("dia", "day") \
    .withColumnRenamed("dia_semana", "day_of_week")

# dim_localizacao
locations = locations \
    .withColumnRenamed("localizacao_id", "location_id") \
    .withColumnRenamed("bairro", "borough") \
    .withColumnRenamed("zona", "zone") \
    .withColumnRenamed("zona_servico", "service_zone")

# dim_clima
weather = weather \
    .withColumnRenamed("clima_id", "weather_id") \
    .withColumnRenamed("data", "date") \
    .withColumnRenamed("precipitacao", "precipitation") \
    .withColumnRenamed("temp_max", "temp_max") \
    .withColumnRenamed("temp_min", "temp_min") \
    .withColumnRenamed("neve", "snow")

print("Colonnes traduites en anglais :")
print(f"  trips:     {trips_raw.columns}")
print(f"  dates:     {dates.columns}")
print(f"  locations: {locations.columns}")
print(f"  weather:   {weather.columns}")

Colonnes traduites en anglais :
  trips:     ['date_id', 'pickup_location_id', 'dropoff_location_id', 'weather_id', 'trip_distance', 'total_amount']
  dates:     ['date_id', 'date', 'year', 'month', 'day', 'day_of_week']
  locations: ['location_id', 'borough', 'zone', 'service_zone']
  weather:   ['weather_id', 'date', 'precipitation', 'temp_max', 'temp_min', 'snow']


In [4]:
# Explorer la structure des donnees
print("=== SCHEMA ===")
trips_raw.printSchema()

print("\n=== APERCU ===")
trips_raw.show(5)

print("\n=== STATISTIQUES ===")
trips_raw.describe().show()

=== SCHEMA ===
root
 |-- date_id: double (nullable = true)
 |-- pickup_location_id: integer (nullable = true)
 |-- dropoff_location_id: integer (nullable = true)
 |-- weather_id: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- total_amount: double (nullable = true)


=== APERCU ===
+-------+------------------+-------------------+----------+-------------+------------+
|date_id|pickup_location_id|dropoff_location_id|weather_id|trip_distance|total_amount|
+-------+------------------+-------------------+----------+-------------+------------+
|  327.0|               168|                159|      NULL|         1.14|        16.8|
|  257.0|                79|                 41|      NULL|         8.46|       52.44|
|  268.0|                70|                 79|      NULL|          9.7|        69.7|
|  171.0|               235|                139|      NULL|         0.61|        12.5|
|   67.0|               160|                232|      NULL|         1.55|        

In [5]:
# Compter les valeurs nulles par colonne
print("=== NULLS PAR COLONNE ===")
trips_raw.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in trips_raw.columns
]).show()

=== NULLS PAR COLONNE ===
+-------+------------------+-------------------+----------+-------------+------------+
|date_id|pickup_location_id|dropoff_location_id|weather_id|trip_distance|total_amount|
+-------+------------------+-------------------+----------+-------------+------------+
|      1|                 0|                  0|    898898|            0|           0|
+-------+------------------+-------------------+----------+-------------+------------+



In [6]:
# Nettoyage
trips_clean = trips_raw \
    .fillna({"weather_id": -1}) \
    .filter(col("trip_distance") > 0) \
    .filter((col("total_amount") > 0) & (col("total_amount") <= 1000)) \
    .dropDuplicates()

import builtins
avant = trips_raw.count()
apres = trips_clean.count()
print(f"Avant nettoyage : {avant}")
print(f"Apres nettoyage : {apres}")
print(f"Lignes supprimees : {avant - apres} ({builtins.round((avant-apres)/avant*100, 1)}%)")

Avant nettoyage : 1000000
Apres nettoyage : 982149
Lignes supprimees : 17851 (1.8%)


In [7]:
# Sauvegarder en Silver (format Parquet)
trips_clean.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/silver/trips_clean/")
dates.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/silver/dates_clean/")
locations.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/silver/locations_clean/")
weather.write.mode("overwrite").parquet("hdfs://namenode:9000/projet/silver/weather_clean/")

print("Bronze -> Silver TERMINE")

Bronze -> Silver TERMINE


In [8]:
spark.stop()